In [7]:
import ollama
import pandas as pd


In [8]:
# from prompt_claude_lite import prompt as base_prompt
from prompt_llama_V2 import prompt as base_prompt

In [ ]:
def process_csv_with_ollama(csv_path, content_column, prompt_template, model_name, output_csv="ollama_outputs.csv"):
    """
    Process a CSV file row by row with an Ollama model.
    
    Args:
        csv_path (str): Path to the CSV file.
        content_column (str): Name of the column containing the text to process.
        prompt_template (str): Prompt template with {{DOCUMENTATION}} placeholder.
        model_name (str): Name of the Ollama model to use.
        output_csv (str): Output CSV file name.
    """
    df = pd.read_csv(csv_path)
    results = []
    for idx, row in df.iterrows():
        doc = str(row[content_column])
        formatted_prompt = prompt_template.replace("{{DOCUMENTATION}}", doc)
        try:
            response = ollama.chat(
                model=model_name,
                messages=[
                    {'role': 'system', 'content': 'You are an expert system for extracting regulatory and technical information from wind farm documents.'},
                    {'role': 'user', 'content': formatted_prompt}
                ]
            )
            output = response['message']['content']
        except Exception as e:
            output = f"Error: {e}"
        results.append({**row, "ollama_output": output})
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_csv, index=False)
    print(f"Processing complete. Results saved to {output_csv}.")

In [ ]:
def process_csv_with_ollama(csv_path, content_column, prompt_template, model_name, output_csv="ollama_outputs.csv"):
    """
    Process a CSV file row by row with an Ollama model.

    Args:
        csv_path (str): Path to the CSV file.
        content_column (str): Name of the column containing the text to process.
        prompt_template (str): Prompt template with {{DOCUMENTATION}} placeholder.
        model_name (str): Name of the Ollama model to use.
        output_csv (str): Output CSV file name.
    """
    df = pd.read_csv(csv_path)
    results = []
    for idx, row in df.iterrows():
        doc = str(row[content_column])
        formatted_prompt = prompt_template.replace("{{DOCUMENTATION}}", doc)
        try:
            response = ollama.chat(
                model=model_name,
                messages=[
                    {'role': 'system', 'content': 'You are an expert system for extracting regulatory and technical information from wind farm documents.'},
                    {'role': 'user', 'content': formatted_prompt}
                ]
            )
            output = response['message']['content']
        except ollama.ResponseError as e:
            output = f"Ollama connection error: {e}"
            print(output)
        except Exception as e:
            output = f"Error: {e}"
            print(output)
        results.append({**row, "ollama_output": output})
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_csv, index=False)
    print(f"Processing complete. Results saved to {output_csv}.")

In [11]:
# --- Prepare Input Data ---
input_1 = """
Offshore wind turbines must adhere to Load Resistance Factor Design (LRFD) principles.
IEC standards currently specify a partial safety factor of 1.35,
but in hurricane-prone areas of the U.S., API standards require additional robustness checks
using a 500-year return period for L2 structures. The discrepancy between IEC and API safety factors
for offshore wind turbines is an ongoing regulatory challenge.
"""

input_2 = """Regulation and compliance in U.S. waters can be under state or federal jurisdiction,
depending on the water body and the distance from shore. State jurisdiction applies to all the Great Lakes’ waters,
and, for most states, three nautical miles seaward (3.5 statute miles or 5.6 kilometers). The exceptions are Louisiana,
Texas, and the Gulf Coast of Florida. Specifically:
Louisiana extends 3 pre-1954 U.S nautical miles (3.455 miles or 5.560 kilometers) seaward.
Texas and the Florida Gulf Coast extend 9 U.S. nautical miles (10.4 miles or 16.7 kilometers) seaward.
"""

input_3 = """
API provides 1-hour, 10-minute, 1-minute, and 3-second wind averages for the Gulf of Mexico.
The 100-year extreme wind and wave conditions govern U.S. oil and gas development. In 2007, BOEM (formerly MMS)
updated its met-ocean criteria as a result of Hurricanes Ivan, Katrina, and Rita (spanning from 2004 to 2005)
when some offshore platforms suffered significant damage. The central section had the highest extreme values,
setting the 100-year 10-minute average mean wind speed at 10 m above water to 54.5 m/s.
"""

# llama2:13b-chat

In [12]:
inputs = [input_1]

for idx, doc in enumerate(inputs, 1):
    # Format the document into the task prompt
    formatted_doc = base_prompt.replace("{{DOCUMENTATION}}", doc)

    response = ollama.chat(
        model='llama2:13b-chat',  # Prefer chat-tuned variant
        messages=[
            {"role": "system", "content": "You are an expert system for extracting regulatory and technical information from wind farm documents."},
            {"role": "user", "content": formatted_doc}
        ]
    )

    print(f"--- Output for input_{idx} ---")
    print(response['message']['content'])
    print("\n")

--- Output for input_1 ---
{
"document_metadata": {
"title": "Offshore wind turbines must adhere to Load Resistance Factor Design (LRFD) principles",
"document_number": null,
"type_of_wind_farm": "Offshore"
},
"regulatory_constraints": [
{
"type": "Safety",
"requirement": "Using a 500-year return period for L2 structures.",
"scope": "Hurricane-prone areas of the U.S.",
"numerical_value": null,
"unit": null,
"source": "API standards",
"related_domains": "Environmental and Technical"
},
{
"type": "Technical",
"requirement": "Adhering to Load Resistance Factor Design (LRFD) principles.",
"scope": null,
"numerical_value": null,
"unit": null,
"source": "IEC standards",
"related_domains": "Environmental and Technical"
}
],
"regulatory_entities": [
{
"entity_name": "API",
"jurisdiction": "Federal",
"role": "Regulatory body"
},
{
"entity_name": "IEC",
"jurisdiction": "International",
"role": "Regulatory body"
}
]
}




# OpenChat

In [13]:
inputs = [input_1]

for idx, doc in enumerate(inputs, 1):
    formatted_prompt = base_prompt.replace("{{DOCUMENTATION}}", doc)
    response = ollama.chat(
        model='openchat',
        messages=[
            {'role': 'system', 'content': 'You are an expert system for extracting regulatory and technical information from wind farm documents.'},
            {'role': 'user', 'content': formatted_prompt}
        ]
    )
    print(f"--- Output for input_{idx} ---")
    print(response['message']['content'])
    print("\n")

--- Output for input_1 ---
 {
  "document_metadata": {
    "title": "Offshore Wind Turbines Load Resistance Factor Design (LRFD)",
    "document_number": null,
    "type_of_wind_farm": "Offshore"
  },
  "regulatory_constraints": [
    {
      "type": "Safety",
      "requirement": "Adhere to Load Resistance Factor Design (LRFD) principles.",
      "scope": null,
      "numerical_value": null,
      "unit": null,
      "source": "Text",
      "related_domains": ["Technical"]
    },
    {
      "type": "Safety",
      "requirement": "Partial safety factor of 1.35 according to IEC standards.",
      "scope": null,
      "numerical_value": "1.35",
      "unit": null,
      "source": "Text",
      "related_domains": ["Technical"]
    },
    {
      "type": "Safety",
      "requirement": "Additional robustness checks using a 500-year return period for L2 structures in hurricane-prone areas of the U.S. according to API standards.",
      "scope": "Hurricane-prone areas of the U.S.",
      "nu

# mistral:7b-instruct

In [14]:
inputs = [input_1]

for idx, doc in enumerate(inputs, 1):
    formatted_prompt = base_prompt.replace("{{DOCUMENTATION}}", doc)
    response = ollama.chat(
        model='mistral:7b-instruct',
        messages=[
            {'role': 'system', 'content': 'You are an expert system for extracting regulatory and technical information from wind farm documents.'},
            {'role': 'user', 'content': formatted_prompt}
        ]
    )
    print(f"--- Output for input_{idx} ---")
    print(response['message']['content'])
    print("\n")

--- Output for input_1 ---
 {
      "document_metadata": {
        "title": "Regulations for Offshore Wind Turbines",
        "document_number": null,
        "type_of_wind_farm": "Offshore"
      },
      "regulatory_constraints": [
        {
          "type": "Safety",
          "requirement": "Offshore wind turbines must adhere to Load Resistance Factor Design (LRFD) principles.",
          "scope": null,
          "numerical_value": null,
          "unit": null,
          "source": "Documentation",
          "related_domains": "Safety"
        },
        {
          "type": "Safety",
          "requirement": "IEC standards specify a partial safety factor of 1.35.",
          "scope": null,
          "numerical_value": "1.35",
          "unit": null,
          "source": "Documentation",
          "related_domains": "Technical"
        },
        {
          "type": "Safety",
          "requirement": "In hurricane-prone areas of the U.S., API standards require additional robustness c

# qwen:7b-chat

In [15]:
inputs = [input_1]

for idx, doc in enumerate(inputs, 1):
    formatted_prompt = base_prompt.replace("{{DOCUMENTATION}}", doc)
    response = ollama.chat(
        model='qwen:7b-chat',
        messages=[
            {'role': 'system', 'content': 'You are an expert system for extracting regulatory and technical information from wind farm documents.'},
            {'role': 'user', 'content': formatted_prompt}
        ]
    )
    print(f"--- Output for input_{idx} ---")
    print(response['message']['content'])
    print("\n")

--- Output for input_1 ---
{  
   "document_metadata": {   
      "title": "Offshore Wind Farm Planning and Regulations",
      "document_number": "WND-FM-P-REG-2023",
      "type_of_wind_farm": "Offshore"
     },  
   "regulatory_constraints": [  
    {  
       "type": "Technical Requirement",  
       "requirement": "The wind turbines must comply with the Load Resistance Factor Design (LRFD) principles, as specified by IEC 61400 series standards.",  
       "scope": "Global, applies to all offshore wind farms",  
       "numerical_value": null,  
       "unit": null,  
       "source": "IEC 61400 standards",  
       "related_domains": ["Technical", "Environmental"]
    },  
    {  
       "type": "Safety Requirement",  
       "requirement": "Wind turbines must be designed and installed in accordance with the relevant safety standards, such as those set by OSHA (U.S.) or equivalent regulatory bodies.",  
       "scope": "United States and other countries with similar regulatory fra

In [16]:
### qwen:14b-chat
### mixtral:8x7b

# Work on CSV

In [17]:
# process_csv_with_ollama("/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Documents/17.csv", "content", base_prompt, "llama2:13b-chat")